# AfriQBench Notebook 02 — Ideal Quantum Benchmark\n\nThis notebook introduces the first quantum-execution layer of AfriQBench. It prepares a compact variational state for the four-qubit transverse-field Ising model (TFIM), evaluates its energy exactly with a statevector, and then reconstructs the Hamiltonian energy from finite-shot Pauli measurements on **Qiskit Aer**.\n\nThe benchmark Hamiltonian is\n\n$$H=-J\\sum_i Z_i Z_{i+1}-h\\sum_i X_i.$$\n\nThe exact classical ground state from Notebook 01 is retained as the scientific reference. The finite-shot estimate is treated as a separate benchmark output with statistical uncertainty.

## 1. Install the quantum extras\n\nFrom the repository root, install the optional quantum dependencies with:\n\n```bash\npython -m pip install -e ".[quantum,notebook]"\n```\n\nThe version ranges are chosen to remain compatible with the current Qiskit/Aer stack used by `metriq-gym`.

In [ ]:
from pathlib import Path\nimport sys\n\nrepo_root = Path.cwd()\nif not (repo_root / 'src').exists():\n    repo_root = repo_root.parent\nsys.path.insert(0, str(repo_root / 'src'))\n\nimport json\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom scipy.optimize import minimize\n\nfrom afriqbench.metrics import absolute_error, relative_error\nfrom afriqbench.models.tfim import tfim_hamiltonian\nfrom afriqbench.quantum.qiskit_tfim import (\n    bind_ansatz,\n    build_hardware_efficient_ansatz,\n    circuit_resource_metrics,\n    estimate_tfim_energy_aer,\n    statevector_energy,\n    statevector_fidelity,\n)\nfrom afriqbench.reference.exact import ground_state\n

## 2. Exact scientific reference\n\nWe first recover the exact reference for the same benchmark instance used in Notebook 01.

In [ ]:
n_qubits = 4\nJ = 1.0\nh = 1.0\nreps = 2\n\nH_exact = tfim_hamiltonian(n_qubits, J=J, h=h, periodic=False)\nexact_energy, exact_state = ground_state(H_exact)\nprint(f'Exact ground-state energy: {exact_energy:.10f}')\n

## 3. Reproducible variational circuit\n\nThe initial AfriQBench ansatz starts from $|+\\rangle^{\\otimes N}$ and alternates $R_y$ rotations with a nearest-neighbour CNOT chain. For `N=4` and two layers, the circuit has eight variational parameters and six raw CNOT gates.

In [ ]:
parameterized_circuit, parameters = build_hardware_efficient_ansatz(\n    n_qubits=n_qubits,\n    reps=reps,\n)\nprint(parameterized_circuit.draw(output='text'))\nprint('Parameter count:', len(parameters))\n

## 4. Deterministic statevector baseline\n\nThe parameter vector below is a reproducible development baseline obtained from a deterministic classical optimization of the two-layer ansatz. It is stored so that regression tests do not depend on rerunning an optimizer.

In [ ]:
baseline_parameters = np.array([\n    0.292792927,\n    -0.0000282104047,\n    -0.100168525,\n    -0.100168480,\n    -0.292808429,\n    -0.518628570,\n    -0.562315096,\n    -0.518620523,\n])\n\nbound_circuit = bind_ansatz(\n    n_qubits=n_qubits,\n    parameter_values=baseline_parameters,\n    reps=reps,\n)\n\nvariational_energy = statevector_energy(\n    bound_circuit, n_qubits, J=J, h=h\n)\nfidelity = statevector_fidelity(bound_circuit, exact_state)\n\nprint(f'Variational statevector energy: {variational_energy:.10f}')\nprint(f'Absolute energy error:           {absolute_error(variational_energy, exact_energy):.10f}')\nprint(f'Relative energy error:           {relative_error(variational_energy, exact_energy):.8e}')\nprint(f'Ground-state fidelity:           {fidelity:.10f}')\n

The reference baseline should be approximately:\n\n- variational energy: **-4.7575478600**;\n- absolute energy error: **0.0012226231**;\n- ground-state fidelity: **0.99976369**.\n\nThese deterministic quantities are useful for testing the benchmark implementation before introducing shot noise or device noise.

## 5. Optional parameter optimization\n\nThe benchmark does not require users to re-optimize the circuit every time. The cell below is included to make the variational procedure transparent. Set `RUN_OPTIMIZATION = True` to refine the baseline parameters locally.

In [ ]:
RUN_OPTIMIZATION = False\n\ndef objective(values):\n    circuit = bind_ansatz(n_qubits, values, reps=reps)\n    return statevector_energy(circuit, n_qubits, J=J, h=h)\n\nif RUN_OPTIMIZATION:\n    optimization = minimize(\n        objective,\n        baseline_parameters,\n        method='COBYLA',\n        options={'maxiter': 500, 'tol': 1e-8},\n    )\n    active_parameters = optimization.x\n    print(optimization)\nelse:\n    active_parameters = baseline_parameters.copy()\n

## 6. Finite-shot ideal simulation with Qiskit Aer\n\nNow we estimate each $ZZ$ and $X$ term from simulated measurements. This is closer to the execution pattern required on real quantum hardware. The result carries statistical uncertainty even though the simulator itself is noiseless.

In [ ]:
benchmark_circuit = bind_ansatz(\n    n_qubits=n_qubits,\n    parameter_values=active_parameters,\n    reps=reps,\n)\n\nshot_result = estimate_tfim_energy_aer(\n    benchmark_circuit,\n    n_qubits=n_qubits,\n    J=J,\n    h=h,\n    shots=20_000,\n    seed=12345,\n)\n\nprint(f"Finite-shot energy: {shot_result['energy']:.8f} ± {shot_result['uncertainty']:.8f}")\nprint(f"Error vs exact:      {absolute_error(shot_result['energy'], exact_energy):.8f}")\n

### Term-level measurements\n\nKeeping the term-level estimates makes the benchmark auditable and allows later hardware runs to identify which observables contribute most strongly to error.

In [ ]:
for term in shot_result['terms']:\n    print(\n        f"{term['term']:>4s}: "\n        f"<{term['term']}> = {term['expectation']:+.6f} "\n        f"± {term['uncertainty']:.6f}"\n    )\n

## 7. Circuit-resource metrics

In [ ]:
resources = circuit_resource_metrics(benchmark_circuit)\nresources\n

These are **raw state-preparation metrics**. When AfriQBench moves to provider-specific execution, the benchmark will also report post-transpilation depth and gate counts because hardware-native compilation can substantially change the resource profile.

## 8. Compare the three energy levels

In [ ]:
labels = ['Exact', 'Variational\nstatevector', 'Finite-shot\nAer']\nvalues = [exact_energy, variational_energy, shot_result['energy']]\n\nfig, ax = plt.subplots(figsize=(7, 4.5))\nbars = ax.bar(labels, values)\nax.set_ylabel('Energy')\nax.set_title('AfriQBench TFIM: exact vs ideal quantum estimates')\nax.axhline(exact_energy, linestyle='--', linewidth=1)\nfor bar, value in zip(bars, values):\n    ax.text(\n        bar.get_x() + bar.get_width() / 2,\n        value,\n        f'{value:.4f}',\n        ha='center',\n        va='top' if value < 0 else 'bottom',\n    )\nfig.tight_layout()\nplt.show()\n

## 9. Save a structured local benchmark record

In [ ]:
record = {\n    'benchmark_id': 'tfim-ideal-qiskit-n4-h1-v0',\n    'model': {\n        'n_qubits': n_qubits,\n        'J': J,\n        'h': h,\n        'boundary': 'open',\n    },\n    'reference': {\n        'ground_state_energy': exact_energy,\n    },\n    'statevector': {\n        'energy': variational_energy,\n        'absolute_error': absolute_error(variational_energy, exact_energy),\n        'fidelity': fidelity,\n    },\n    'finite_shot_aer': shot_result,\n    'raw_circuit_resources': resources,\n}\n\nresults_dir = repo_root / 'results'\nresults_dir.mkdir(exist_ok=True)\noutput_path = results_dir / 'ideal_quantum_n4_h1_run.json'\noutput_path.write_text(json.dumps(record, indent=2), encoding='utf-8')\nprint(output_path)\n

## What this adds to AfriQBench\n\nNotebook 01 established trusted classical reference values. Notebook 02 adds a reproducible quantum state-preparation circuit, a deterministic statevector check, finite-shot Pauli measurement, uncertainty estimation, and circuit-resource reporting.\n\nThe next benchmark layer will introduce controlled noise and compare the noisy result with the exact and ideal baselines before moving to provider-specific hardware.